# Boundary Element Model

The Boundary Element Model (BEM) defines the different compartments of the head (brain, skull, and scalp) by generating three distinct surfaces:
- the inner skull
- the outer skull
- the outer scalp.

The BEM is constructed by combining these surfaces, with specified electrical conductivities between the layers. These conductivities are typically assumed to be homogeneous and isotropic within each compartment. <br>

We first generate the surfaces by segmenting the T1-weighted MRI using FreeSurfer’s watershed algorithm. Ideally, the resulting surfaces should be concentric; however, small inaccuracies can occur and must be corrected before proceeding. This correction is performed using our `correct_skull` function. Finally, the corrected surfaces are assembled by assigning the electrical conductivities that define each inter-compartment region.

## 📦 Imports

In [1]:
import mindmap as mm
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**

**Anatomy**
- FREESURFER reconstruction folder.
    - `T1`: T1-weigthed MRI (freesurfer space)

🎚️ **<font color=pink>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)

In [2]:
# ------------ Parameters -------------
fconfig = "../../papers/simf/configuration.yml"
sub     = 1
# -------------------------------------

config = mm.io.read_config(fconfig)
sub    = config["SUBS"][sub-1]

## ⚙️ 1. Create surfaces

📤 **<font color=pink>Outputs</font>** (files in freesurfer reconstruction bem folder)
- `outer_skin.surf`: outer skin surface (scalp)
- `outer_skull_raw.surf`: outer skull surface
- `inner_skull.surf`: inner skull surface
- `brain.surf`: copy of brain surface
- `inner_skull_raw.surf`: if inner skull going out of outer skull (watershed bug), push outside vertices back inside. This file is the original (bugged) surface
- `inner_skull_changes.vtk`: shows changes between raw inner skull and corrected inner skull

In [3]:
mne.bem.make_watershed_bem(
    subject      = sub,
    subjects_dir = config["FS_DIR"],
    volume       = "T1",
    overwrite    = True,
    verbose      = "warning",
)

## ⚙️ 2. Correct surfaces

📥 **<font color=pink>Inputs</font>**

**Anatomy**
- bem surfaces

📤 **<font color=pink>Outputs</font>** (files in freesurfer reconstruction bem folder)
- corrected bem surfaces

🎚️ **<font color=pink>Configuration</font>**:
- `min_skull_thick`: Minimum skull thickness in [cm] to apply during correction.

**Notes:**
Correct skull sends and "ERROR:root:No points/cells to operate on". The program will continue without stopping.

In [4]:
# ------------ Parameters -------------
min_skull_thick = 2
# -------------------------------------

mm.head.correct_skull(
    subject     = sub,
    offset      = min_skull_thick,
    fs_dir      = config["FS_DIR"],
    save_arrows = True,
)

## ⚙️ 3. Assemble the model
In this section, we apply topological checks to verify if the surfaces are compatible with each other and apply electrical conductivities to each compartment.

📤 **<font color=pink>Outputs</font>**
- Conductor Model (`head_model`)

🎚️ **<font color=pink>Boundary Element Model</font>**:
- `conductivities`: The 3 conductivities indicating the electrical conductivity in [S/cm] in each compartment.
    - conductivity of the brain tissue.
    - conductivity of the skull tissue.
    - conductivity of the skin tissue.
- `resolution`: The resolution can be optimized for various subsampling level (`None`, `ico5`=20484, `ico4`=5120, `ico3`=1280). Subsampling using `ico5` or `None` contains to many datapoints and is unable to save. <br>

In [5]:
############# Parameters ################
# properties
conductivities = [0.3, 0.006, 0.3]
resolution     = 4 
#########################################

model = mne.make_bem_model(
        subject      = sub,
        subjects_dir = config["FS_DIR"],
        conductivity = conductivities,
        ico          = resolution,
        verbose      = "warning",
)

bem  = mne.make_bem_solution(model, verbose="warning")

💾 **Save BEM** <br>
🎚️ **<font color=pink>Options</font>**:
- `overwrite`: Overwrite head model if already exists.
- `purpose`: Purpose of model (`inv` for solving inverse problem, `sim` for generating simulations). Will adapt the naming of the file.

In [9]:
# ------------ Parameters -------------
overwrite = True
purpose   = "inv"
# -------------------------------------

if purpose == "inv":
    fbem = config["MM_DIR"]/sub/"fwd"/f"{sub}_desc-inverse_bem.fif"
elif purpose == "sim":
    fbem = config["MM_DIR"]/sub/"fwd"/f"{sub}_desc-simulation_bem.fif"
else:
    raise ValueError(f"Uknown purpose '{purpose}'")

if not fbem.parent.exists():
    fbem.parent.mkdir(parents=True)

mne.write_bem_solution(fbem, bem, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved {purpose} BEM solution for subject {sub}\033[0m")


Successfully saved inv BEM solution for subject vepcon_sub-10
